# Session 9 — Deploying Automated Machine Learning (AutoML) Services using AWS SageMaker

**Goal:** the AWS counterpart to Session 4's Vertex AI AutoML — use
**SageMaker Autopilot** to search over models and hyperparameters automatically, then
deploy the winner, without writing a training script like Session 8 required.

## Autopilot vs. hand-written SageMaker training (Session 8)

Session 8's `sagemaker_train.py` is a training script *you* wrote — you chose
`RandomForestClassifier` and its hyperparameters. Autopilot instead explores multiple
algorithms (linear models, trees, ensembles) and their hyperparameters for you,
returning a leaderboard of candidate pipelines ranked by your chosen metric.

## Prerequisites

Needs an **AWS account** with SageMaker access — not available in this sandbox.
Complete, correct reference code below.

```bash
pip install boto3 sagemaker
```

In [ ]:
import boto3
import sagemaker
from sagemaker import AutoML

REGION = "us-east-1"
BUCKET = "your-sagemaker-bucket"
ROLE = sagemaker.get_execution_role()

## Step 1 — Upload training data to S3

Autopilot reads a single CSV with the target column included (here, the heart
disease dataset used across this course's MLOps module).

In [ ]:
session = sagemaker.Session()
s3_train_path = session.upload_data(
    path="heart_disease_train.csv",
    bucket=BUCKET,
    key_prefix="autopilot/heart-disease",
)
print("Uploaded to:", s3_train_path)

## Step 2 — Configure and launch an Autopilot job

`target_attribute_name` is the label column; `problem_type` and `job_objective` tell
Autopilot what kind of problem this is and what to optimize for — everything else
(feature engineering, algorithm selection, hyperparameter search) is automatic.

In [ ]:
automl = AutoML(
    role=ROLE,
    target_attribute_name="target",
    output_path=f"s3://{BUCKET}/autopilot/output",
    problem_type="BinaryClassification",
    job_objective={"MetricName": "AUC"},
    max_candidates=20,          # cap on how many pipelines Autopilot tries
    total_job_runtime_in_seconds=3600,
)

automl.fit(inputs=s3_train_path, job_name="heart-disease-autopilot")
print("Autopilot job launched.")

## Step 3 — Inspect the candidate leaderboard

Every candidate pipeline Autopilot tried, ranked by the objective metric — the AWS
equivalent of Session 1's `mlflow.search_runs` leaderboard, but the candidates
themselves (not just hyperparameters) were chosen automatically.

In [ ]:
candidates = automl.list_candidates(sort_by="FinalObjectiveMetricValue", sort_order="Descending")
for i, candidate in enumerate(candidates[:5]):
    metric = candidate["FinalAutoMLJobObjectiveMetric"]["Value"]
    print(f"{i+1}. {candidate['CandidateName']}  AUC={metric:.4f}")

## Step 4 — Deploy the best candidate

Same shape as every other "deploy to a managed endpoint" step in this course
(Sessions 4 and 8): one call provisions an autoscaling HTTPS endpoint.

In [ ]:
best_candidate = automl.best_candidate()
print("Best candidate:", best_candidate["CandidateName"])

predictor = automl.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    candidate=best_candidate,
    endpoint_name="heart-disease-autopilot-endpoint",
)
print(f"Deployed endpoint: {predictor.endpoint_name}")

## Step 5 — Get a prediction

In [ ]:
sample_row = "58,1,0,128,216,0,0,131,1,2.2,1,3,3"
result = predictor.predict(sample_row)
print("Prediction:", result)

## Step 6 — Explainability: Autopilot's built-in feature importance

Autopilot automatically runs a SHAP-based explainability report for the best
candidate (deep-dived by hand in Session 22) as part of the job.

In [ ]:
explainability_report_uri = automl.describe_auto_ml_job()["BestCandidate"] \
    .get("CandidateProperties", {}) \
    .get("CandidateArtifactLocations", {}) \
    .get("Explainability")
print("Explainability report location:", explainability_report_uri)
print("Download and open the report.json/report.html from that S3 location to view it.")

## Step 7 — Clean up

In [ ]:
predictor.delete_endpoint()
print("Endpoint deleted -- billing stopped.")

## What to try next

* Compare Autopilot's best AUC against the hand-tuned RandomForest from Session 8 on
  the same dataset — Autopilot's advantage grows with dataset complexity.
* Session 19 wraps SageMaker training + Autopilot + deployment into one orchestrated
  pipeline (SageMaker Pipelines) instead of running each step as a standalone notebook
  cell.